# LLM 7 — Agents: AI that uses tools, carefully

A chatbot talks. An **agent** acts: the model can ask YOUR CODE to run a tool, see the result, and continue. You'll build the loop yourself — no magic.

> **Working with your AI pair:** paste any error into your AI and ask *"explain this error like I'm new, then help me fix it — don't just give me the answer."*


In [ ]:
%pip install -q anthropic
import os, anthropic
from getpass import getpass
os.environ.setdefault('ANTHROPIC_API_KEY', getpass('Class API key: '))
MODEL='claude-opus-5'
client=anthropic.Anthropic()


## 1. Define a tool — a function plus a description the model reads


In [ ]:
def practice_schedule(team):
    data = {'soccer':'Mon/Wed 4pm, Douglass Park','robotics':'Tue/Thu 4pm, room 214',
            'debate':'Fri 3:30pm, library'}
    return data.get(team.lower(), f'no schedule found for {team}')

TOOLS = [{
  'name': 'practice_schedule',
  'description': 'Look up the real practice schedule for a school team. Use this instead of guessing.',
  'input_schema': {'type':'object','properties':{'team':{'type':'string'}},'required':['team']}
}]


## 2. The agent loop — request, run, return, repeat


In [ ]:
def agent(question):
    messages=[{'role':'user','content':question}]
    while True:
        r = client.messages.create(model=MODEL, max_tokens=500,
                                   tools=TOOLS, messages=messages)
        if r.stop_reason == 'tool_use':
            messages.append({'role':'assistant','content':r.content})
            results=[]
            for block in r.content:
                if block.type=='tool_use':
                    print(f'  [model wants {block.name}({block.input})]')
                    out = practice_schedule(**block.input)
                    results.append({'type':'tool_result','tool_use_id':block.id,'content':out})
            messages.append({'role':'user','content':results})
        else:
            return ''.join(b.text for b in r.content if b.type=='text')

print(agent('When does the robotics team practice, and will that clash with debate?'))


## 3. Read what happened
The model DECIDED to call your function — maybe twice — read the real answers, and only then replied. That decision loop is every AI agent in the world, from this cell to the ones that write software.


## 4. 'Carefully' is half the lesson title
Your function was harmless. Real agents get tools that send email, spend money, delete files. The rules: give an agent the SMALLEST tool that works · log every call (the print line!) · put a human between the agent and anything irreversible. You'll live these rules in the nonprofit work someday.


## 5. Your turn
Add a second tool — `room_lookup(club)` returning meeting rooms — and ask a question needing BOTH tools. Watch the printed call log.


In [ ]:
# your second tool


## 6. The build
**An agent with one real tool of your design** (fake data is fine; real decisions required): schedules, menu, bus times, library hours. Demo: one question it answers with the tool, one where it says the tool can't help. **Turn-in:** the call log of both.
